# Implementasi DBSCAN pada Katalog Gempa BMKG

Notebook ini adalah **lanjutan** dari *Final_Preprocessing_DBSCAN_Gempa.ipynb*. Input-nya adalah set model hasil preprocessing (`gempa_siap_dbscan.csv`: 14.347 gempa, M ≥ 4,5, periode 2010 s.d. 2026, wilayah Indonesia dan sekitarnya).

| No | Tugas | Bagian |
|---|---|---|
| 1 | Konsep epsilon dan MinPts | Bagian 1 |
| 2 | Kandidat epsilon dengan k-distance graph | Bagian 4 |
| 3 | Penentuan MinPts beserta alasannya | Bagian 5 |
| 4 | DBSCAN dengan Scikit-learn | Bagian 6 |
| 5 | Label cluster setiap data | Bagian 7 |
| 6 | Jumlah cluster dan noise | Bagian 8 |
| 7 | Visualisasi hasil | Bagian 9 |
| 8 | Perbandingan kombinasi parameter | Bagian 10 |

**Prinsip:** tidak ada parameter yang diklaim "paling optimal" tanpa pengujian. Nilai awal (*baseline*) hanya titik awal yang bisa dipertanggungjawabkan, lalu dibandingkan dengan kombinasi lain di Bagian 10.

**Cara pakai di Colab:** jalankan sel dari atas. Jika sesi preprocessing masih aktif, variabel `model` dipakai langsung. Jika tidak, sel pemuatan akan mencari `gempa_siap_dbscan.csv` atau meminta Anda mengunggahnya.

## 1. Konsep parameter epsilon (ε) dan MinPts

DBSCAN (*Density-Based Spatial Clustering of Applications with Noise*) mendefinisikan cluster sebagai **daerah padat yang dipisahkan oleh daerah jarang**. Hanya ada dua parameter:

**Epsilon (ε, `eps`)** adalah radius lingkungan (*neighborhood*) di sekitar sebuah titik. Pada notebook ini jaraknya adalah jarak busur bumi (haversine), sehingga ε dinyatakan dalam **km** dan dikonversi ke radian dengan `eps_rad = eps_km / 6371,0088` karena scikit-learn meminta radian untuk metrik haversine.

**MinPts (`min_samples` di scikit-learn)** adalah jumlah minimum titik di dalam radius ε, **termasuk titik itu sendiri**, agar daerah tersebut dianggap padat.

Dari dua parameter itu setiap titik mendapat satu dari tiga status:

| Status | Definisi |
|---|---|
| **Core point** | Ada ≥ MinPts titik (termasuk dirinya) dalam radius ε |
| **Border point** | Bukan core, tetapi berada dalam radius ε dari suatu core point |
| **Noise** (label `-1`) | Bukan core dan bukan border |

Cluster dibentuk dengan menghubungkan core point yang saling berjarak ≤ ε (*density-connected*), lalu border point ditempelkan ke cluster core yang menjangkaunya.

**Pengaruh nilai parameter**

| Parameter | Terlalu kecil | Terlalu besar |
|---|---|---|
| ε | Hampir semua titik menjadi noise, cluster terpecah-pecah | Cluster yang seharusnya terpisah menyatu (*chaining*), akhirnya satu cluster raksasa |
| MinPts | Noise ikut membentuk cluster; mudah terjadi chaining | Hanya daerah sangat padat yang lolos; banyak titik menjadi noise |

**Keduanya bekerja bersama sebagai ambang kepadatan.** Dengan pendekatan bidang datar (cukup baik untuk ε kecil), ambang kepadatan kira-kira `MinPts / (π · ε²)` kejadian per km². Menaikkan ε sambil mempertahankan MinPts berarti **menurunkan** ambang kepadatan, dan sebaliknya.

**Keterbatasan yang relevan untuk data gempa:** DBSCAN memakai satu ambang kepadatan global. Seismisitas Indonesia sangat tidak merata (misalnya Laut Banda jauh lebih padat daripada Kalimantan), sehingga satu pasang parameter bisa pas untuk zona padat tetapi terlalu ketat atau terlalu longgar untuk zona lain. Itu sebabnya Bagian 10 menguji banyak kombinasi, dan cluster hasilnya perlu ditafsirkan sebagai **kelompok kepadatan spasial**, bukan otomatis sebagai satu sesar atau satu rangkaian gempa susulan.

## 2. Impor pustaka dan konfigurasi

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

# ===================== KONFIGURASI =====================
INPUT_CSV = None        # contoh: "/content/gempa_siap_dbscan.csv"; None = cari otomatis / minta unggah
OUT_DIR   = "/content/hasil_dbscan" if IN_COLAB else "./hasil_dbscan"
os.makedirs(OUT_DIR, exist_ok=True)

EARTH_RADIUS_KM = 6371.0088
RANDOM_STATE    = 42

# --- Bagian 4-5: k-distance dan MinPts ---
MINPTS_CANDIDATES = [4, 10, 15, 20]   # nilai MinPts yang digambar k-distance-nya
MINPTS_MAIN       = 10                # MinPts untuk baseline (alasan: Bagian 5); harus ada di daftar di atas
EPS_BASELINE_KM   = None              # None = pakai titik siku (knee) k-distance; isi angka (km) untuk menimpa
KNEE_TRIM_PCT     = 99                # titik siku dicari pada 99% data terendah (ekor outlier dibuang dari pencarian)

# --- Bagian 10: grid perbandingan ---
EPS_GRID_KM   = [5, 10, 15, 20, 25, 30, 40, 50]
MINPTS_GRID   = [4, 10, 15, 20]
SILHOUETTE_SAMPLE = 5000              # jumlah titik sampel untuk silhouette (hemat memori)

# --- Tampilan ---
SHOW_ALL_LABELS = False               # True = tampilkan label SEMUA baris di layar (14 ribu baris)
print("Berjalan di Google Colab:", IN_COLAB)

## 3. Memuat data hasil preprocessing

Fitur clustering hanya `latitude` dan `longitude`, dikonversi ke **radian** untuk metrik haversine (sesuai keputusan di notebook preprocessing; `StandardScaler` sengaja tidak dipakai karena merusak geometri bumi). Kolom lain (magnitudo, kedalaman, lokasi, waktu) hanya dipakai untuk menafsirkan hasil.

In [ ]:
def load_model_data():
    g = globals()
    # 1) Sesi preprocessing yang sama: DataFrame `model` masih ada
    if isinstance(g.get("model"), pd.DataFrame) and {"latitude", "longitude"} <= set(g["model"].columns):
        print("Memakai DataFrame `model` dari notebook preprocessing (sesi yang sama).")
        return g["model"].copy()
    # 2) Cari file CSV hasil preprocessing
    candidates = [INPUT_CSV,
                  "/content/hasil_preprocessing/gempa_siap_dbscan.csv",
                  "/content/gempa_siap_dbscan.csv", "gempa_siap_dbscan.csv",
                  "/content/drive/MyDrive/gempa_siap_dbscan.csv"]
    for c in candidates:
        if c and os.path.exists(c):
            print("Membaca:", c)
            return pd.read_csv(c)
    # 3) Minta unggah (Colab)
    if IN_COLAB:
        from google.colab import files
        print("File tidak ditemukan. Unggah gempa_siap_dbscan.csv (hasil notebook preprocessing)...")
        up = files.upload()
        return pd.read_csv(next(iter(up)))
    raise FileNotFoundError("Isi INPUT_CSV dengan lokasi gempa_siap_dbscan.csv")

data = load_model_data()
if "datetime" in data.columns:
    data["datetime"] = pd.to_datetime(data["datetime"], utc=True, format="ISO8601")
    data = data.sort_values("datetime", kind="stable")
data = data.reset_index(drop=True)

# --- Validasi ---
assert data[["latitude", "longitude"]].notna().all().all(), "Ada koordinat kosong"
assert data["latitude"].between(-90, 90).all() and data["longitude"].between(-180, 180).all()

# Fitur: (lat, lon) dalam radian, urutan sesuai scikit-learn untuk haversine
X = np.radians(data[["latitude", "longitude"]].to_numpy())
N = len(X)

def eps_rad(eps_km):
    return eps_km / EARTH_RADIUS_KM

print(f"Jumlah data : {N:,}")
print(f"Fitur X     : {X.shape} (radian)")
print(f"Lintang     : {data.latitude.min():.2f} s.d. {data.latitude.max():.2f}")
print(f"Bujur       : {data.longitude.min():.2f} s.d. {data.longitude.max():.2f}")
display(data.head())

## 4. Kandidat epsilon dengan k-distance graph

**Logika.** Untuk setiap titik, hitung jarak ke tetangga ke-*k*. Urutkan semua jarak itu dari kecil ke besar. Pada kurva ini:
- bagian landai di kiri = titik di daerah padat,
- bagian yang menanjak tajam di kanan = titik terpencil (calon noise),
- **"siku" (knee)** tempat kurva mulai menanjak tajam adalah kandidat ε: di bawahnya titik masih punya *k* tetangga dekat, di atasnya tidak.

**Detail yang menentukan ketepatan.** `min_samples` di scikit-learn menghitung titik itu sendiri. Saat kita menanyakan tetangga untuk titik yang ada di data latih, tetangga pertama adalah dirinya sendiri (jarak 0). Maka untuk `min_samples = m`, jarak yang relevan adalah **jarak ke tetangga ke-*m* termasuk dirinya**, yaitu kolom ke-*m* hasil `kneighbors`. Sebuah titik adalah core point tepat ketika jarak ini ≤ ε. Konsekuensi praktisnya: **persentil ke-*p* dari k-distance berarti kira-kira *p*% titik akan menjadi core point** pada ε tersebut.

Kurva dihitung untuk beberapa nilai *k* sekaligus karena MinPts baru diputuskan di Bagian 5.

**Peringatan jujur:** katalog gempa adalah data dengan kepadatan yang berubah mulus (zona subduksi menerus), sehingga siku bisa tidak tajam atau tidak unik. Karena itu titik siku di sini diperlakukan sebagai **satu kandidat**, bukan jawaban. Pengujian sebenarnya ada di Bagian 10.

In [ ]:
K_MAX = max(MINPTS_CANDIDATES)
nn = NearestNeighbors(n_neighbors=K_MAX, metric="haversine", algorithm="ball_tree").fit(X)
dist_rad, _ = nn.kneighbors(X)               # kolom 0 = titik itu sendiri (jarak 0)
dist_km = dist_rad * EARTH_RADIUS_KM

kdist = {k: np.sort(dist_km[:, k - 1]) for k in MINPTS_CANDIDATES}   # kolom ke-k = tetangga ke-k termasuk diri sendiri

def knee_index(y, trim_pct=99):
    """Titik siku = titik kurva terjauh di bawah garis lurus yang menghubungkan ujung-ujung kurva
    (kurva dinormalisasi ke [0,1]; ekor outlier di atas persentil trim_pct tidak ikut dicari)."""
    y = np.asarray(y)[: int(len(y) * trim_pct / 100)]
    if y[-1] == y[0]:
        return len(y) - 1
    xn = np.linspace(0, 1, len(y))
    yn = (y - y[0]) / (y[-1] - y[0])
    return int(np.argmax(xn - yn))

rows = []
for k in MINPTS_CANDIDATES:
    y = kdist[k]; ki = knee_index(y, KNEE_TRIM_PCT)
    rows.append({"MinPts (k)": k,
                 "p50 (km)": np.percentile(y, 50), "p75 (km)": np.percentile(y, 75),
                 "p90 (km)": np.percentile(y, 90), "p95 (km)": np.percentile(y, 95),
                 "knee (km)": y[ki], "knee = persentil ke": 100 * (ki + 1) / len(y)})
cand = pd.DataFrame(rows).set_index("MinPts (k)").round(2)
print("Kandidat epsilon (km) dari k-distance. Persentil p berarti ~p% titik menjadi core point:")
display(cand)

# --- Grafik ---
colors = plt.cm.viridis(np.linspace(0.05, 0.85, len(MINPTS_CANDIDATES)))
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
for c, k in zip(colors, MINPTS_CANDIDATES):
    y = kdist[k]; ki = knee_index(y, KNEE_TRIM_PCT)
    for a in ax:
        a.plot(np.arange(N), y, color=c, lw=1.3, label=f"k = {k}")
        a.scatter([ki], [y[ki]], color=c, edgecolor="k", zorder=5, s=35)
ax[0].set_title("k-distance graph (seluruh data)")
ax[1].set_title("Diperbesar: 0 s.d. persentil ke-99")
ax[1].set_ylim(0, max(np.percentile(kdist[k], 99) for k in MINPTS_CANDIDATES) * 1.05)
for a in ax:
    a.set_xlabel("Titik data (diurutkan menurut k-distance)"); a.set_ylabel("Jarak ke tetangga ke-k (km)")
    a.grid(alpha=.25)
ax[0].legend(frameon=False, title="Lingkaran = titik siku")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/k_distance_graph.png", dpi=130); plt.show()

## 5. Penentuan MinPts

**Dasar yang dipakai**

1. **Batas bawah teoretis.** MinPts harus ≥ D + 1, dengan D = jumlah dimensi fitur (Ester dkk., 1996). Untuk koordinat 2D, itu 3. Aturan praktis umum (Sander dkk., 1998) adalah **MinPts ≥ 2·D = 4**.
2. **Ukuran data.** Dengan N = 14.347, nilai sekecil 4 sangat mudah terpenuhi oleh pasangan atau segitiga gempa yang kebetulan berdekatan, sehingga cluster mudah tersambung lewat jembatan tipis (chaining). Heuristik yang kadang dipakai, MinPts ≈ ln(N), memberi sekitar 10.
3. **Makna seismologis.** Data berisi gempa M ≥ 4,5 yang sudah difilter kelengkapan katalognya. Menuntut ≥ 10 kejadian (termasuk dirinya) dalam radius ε berarti sebuah "kelompok" setidaknya setara rangkaian susulan atau *swarm* kecil, bukan kebetulan dua-tiga kejadian.
4. **Tidak terlalu besar.** Nilai besar (mis. 20+) menjadikan zona dengan seismisitas rendah (Kalimantan, Jawa bagian utara) seluruhnya noise.

**Keputusan: `MINPTS_MAIN = 10` sebagai baseline**, dengan catatan eksplisit bahwa ini **keputusan yang bisa dipertanggungjawabkan, bukan nilai yang terbukti terbaik**. Nilai 4, 15, dan 20 tetap diuji di Bagian 10.

In [ ]:
D = X.shape[1]
heur = pd.DataFrame({
    "Aturan": ["Batas bawah D + 1", "Aturan praktis 2 x D", "Heuristik ln(N)", "Dipilih sebagai baseline"],
    "Nilai":  [D + 1, 2 * D, round(float(np.log(N)), 2), MINPTS_MAIN]})
display(heur)
assert MINPTS_MAIN in MINPTS_CANDIDATES, "MINPTS_MAIN harus ada di MINPTS_CANDIDATES"

# --- Epsilon baseline: titik siku k-distance untuk MinPts terpilih (kecuali ditimpa manual) ---
y_main = kdist[MINPTS_MAIN]
knee_km = float(y_main[knee_index(y_main, KNEE_TRIM_PCT)])
EPS_BASELINE = round(knee_km, 1) if EPS_BASELINE_KM is None else float(EPS_BASELINE_KM)
core_share = float((y_main <= EPS_BASELINE).mean() * 100)

print(f"\nBaseline: MinPts = {MINPTS_MAIN}, eps = {EPS_BASELINE} km "
      f"({'titik siku k-distance' if EPS_BASELINE_KM is None else 'diisi manual'})")
print(f"Pada eps ini ~{core_share:.1f}% titik memenuhi syarat core point.")
print("Ini BASELINE untuk Bagian 6-9, bukan klaim optimal. Pembandingan ada di Bagian 10.")

## 6. Implementasi DBSCAN (Scikit-learn)

Konfigurasi yang dipakai: `metric="haversine"` pada koordinat radian, `algorithm="ball_tree"` (struktur data yang mendukung haversine dan efisien untuk data 2D), dan `eps` dalam radian.

Fungsi `fit_dbscan` dibuat reusable agar kombinasi lain dapat dijalankan dengan satu baris.

In [ ]:
def fit_dbscan(eps_km, min_samples):
    db = DBSCAN(eps=eps_rad(eps_km), min_samples=int(min_samples),
                metric="haversine", algorithm="ball_tree", n_jobs=-1)
    t0 = time.time()
    labels = db.fit_predict(X)
    return db, labels, time.time() - t0

db_base, labels_base, sec = fit_dbscan(EPS_BASELINE, MINPTS_MAIN)
print(f"DBSCAN selesai dalam {sec:.1f} detik")
print(f"Parameter : eps = {EPS_BASELINE} km (= {eps_rad(EPS_BASELINE):.6f} rad), min_samples = {MINPTS_MAIN}")
print(f"Metrik    : haversine | algoritma: ball_tree")

## 7. Label cluster setiap data

Label `0, 1, 2, ...` adalah nomor cluster (urutan penemuan, **tidak mengandung makna peringkat**), dan **`-1` = noise**. Kolom `status` menunjukkan apakah titik itu core, border, atau noise. Label seluruh baris disimpan di CSV.

In [ ]:
def attach_labels(df, labels, db):
    out = df.copy()
    out["cluster"] = labels
    status = np.full(len(out), "border", dtype=object)
    status[db.core_sample_indices_] = "core"
    status[labels == -1] = "noise"
    out["status"] = status
    return out

result = attach_labels(data, labels_base, db_base)
show_cols = [c for c in ["eventID", "datetime", "latitude", "longitude", "magnitude", "location", "cluster", "status"]
             if c in result.columns]

print("25 baris pertama:")
display(result[show_cols].head(25))
print("10 baris terakhir:")
display(result[show_cols].tail(10))

if SHOW_ALL_LABELS:
    with pd.option_context("display.max_rows", None):
        display(result[show_cols])

print("Jumlah titik per label (maks 15 teratas; -1 = noise):")
display(result["cluster"].value_counts().head(15).rename_axis("label").to_frame("jumlah"))

result.to_csv(f"{OUT_DIR}/hasil_dbscan_baseline.csv", index=False)
print(f"Label SEMUA {len(result):,} baris tersimpan di: {OUT_DIR}/hasil_dbscan_baseline.csv")

## 8. Jumlah cluster dan noise

In [ ]:
def cluster_stats(labels):
    ids, cnt = np.unique(labels[labels != -1], return_counts=True)
    n_noise = int((labels == -1).sum())
    largest = int(cnt.max()) if len(cnt) else 0
    return {"n_clusters": int(len(ids)), "n_noise": n_noise,
            "noise_pct": round(100 * n_noise / len(labels), 2),
            "n_clustered": int(len(labels) - n_noise),
            "largest_cluster": largest,
            "largest_pct": round(100 * largest / len(labels), 2),
            "median_cluster_size": float(np.median(cnt)) if len(cnt) else 0.0}

st = cluster_stats(labels_base)
n_core = int(len(db_base.core_sample_indices_))
n_border = st["n_clustered"] - n_core

print(f"Jumlah cluster (tanpa noise) : {st['n_clusters']}")
print(f"Jumlah noise (label -1)      : {st['n_noise']:,} ({st['noise_pct']}%)")
print(f"Titik dalam cluster          : {st['n_clustered']:,} (core {n_core:,} + border {n_border:,})")
print(f"Cluster terbesar             : {st['largest_cluster']:,} titik ({st['largest_pct']}% dari seluruh data)")
print(f"Median ukuran cluster        : {st['median_cluster_size']:.0f} titik")

if st["n_clusters"] == 0:
    print("\nPERHATIAN: tidak ada cluster. eps terlalu kecil untuk MinPts ini; lihat Bagian 10.")
elif st["largest_pct"] > 50:
    print("\nPERHATIAN: satu cluster memuat >50% seluruh data. Indikasi chaining (eps terlalu besar / MinPts terlalu kecil); bandingkan dengan Bagian 10.")

## 9. Visualisasi hasil clustering

Tiga tampilan: (a) peta cluster, (b) peta status core / border / noise, (c) ukuran cluster terbesar, serta tabel profil cluster. Warna cluster dipilih menurut ukuran: **20 cluster terbesar mendapat warna berbeda**, cluster yang lebih kecil memakai warna berulang, jadi dua cluster berwarna sama belum tentu satu cluster.

In [ ]:
PALETTE = plt.cm.tab20(np.arange(20))

def color_map(labels):
    ids, cnt = np.unique(labels[labels != -1], return_counts=True)
    order = ids[np.argsort(-cnt)]
    return {cid: PALETTE[i % 20] for i, cid in enumerate(order)}

def plot_clusters(labels, title, ax, s_cluster=6):
    cm = color_map(labels)
    noise = labels == -1
    ax.scatter(data.longitude[noise], data.latitude[noise], s=3, c="lightgray", alpha=.6, label="noise", rasterized=True)
    if (~noise).any():
        cols = np.array([cm[l] for l in labels[~noise]])
        ax.scatter(data.longitude[~noise], data.latitude[~noise], s=s_cluster, c=cols, rasterized=True)
    s = cluster_stats(labels)
    ax.set_title(f"{title}\n{s['n_clusters']} cluster, {s['n_noise']:,} noise ({s['noise_pct']}%)", fontsize=9)
    ax.set_xlabel("Bujur"); ax.set_ylabel("Lintang"); ax.set_aspect("equal")

# (a) peta cluster
fig, ax = plt.subplots(figsize=(11, 6.5))
plot_clusters(labels_base, f"DBSCAN baseline (eps = {EPS_BASELINE} km, MinPts = {MINPTS_MAIN})", ax)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/peta_cluster_baseline.png", dpi=130); plt.show()

# (b) core / border / noise  dan  (c) ukuran cluster
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1.5, 1]})
sty = {"core": ("#1f77b4", 5), "border": ("#ff7f0e", 5), "noise": ("#bbbbbb", 3)}
for name, (c, s) in sty.items():
    m = result["status"] == name
    ax[0].scatter(result.longitude[m], result.latitude[m], s=s, c=c, label=f"{name} ({int(m.sum()):,})", rasterized=True,
                  alpha=.8 if name != "noise" else .5)
ax[0].set_title("Status titik: core / border / noise"); ax[0].set_xlabel("Bujur"); ax[0].set_ylabel("Lintang")
ax[0].set_aspect("equal"); ax[0].legend(frameon=False, markerscale=2)

sizes = result.loc[result.cluster != -1, "cluster"].value_counts().head(15)
if len(sizes):
    cm = color_map(labels_base)
    ax[1].barh([str(i) for i in sizes.index[::-1]], sizes.values[::-1], color=[cm[i] for i in sizes.index[::-1]])
    ax[1].set_xlabel("Jumlah gempa"); ax[1].set_ylabel("Label cluster")
    ax[1].set_title(f"{len(sizes)} cluster terbesar")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/status_dan_ukuran_cluster.png", dpi=130); plt.show()

# --- Profil cluster (untuk tafsir) ---
def cluster_profile(res, top=15):
    g = res[res.cluster != -1].groupby("cluster")
    agg = {"jumlah": ("cluster", "size"), "lat_rata2": ("latitude", "mean"), "lon_rata2": ("longitude", "mean"),
           "mag_rata2": ("magnitude", "mean"), "mag_maks": ("magnitude", "max")}
    if "depth" in res.columns:
        agg["kedalaman_median_km"] = ("depth", "median")
    if "location" in res.columns:
        agg["lokasi_tersering"] = ("location", lambda s: s.mode().iat[0] if len(s.mode()) else "")
    prof = g.agg(**agg)
    if "datetime" in res.columns:
        prof["mulai"] = g["datetime"].min().dt.date
        prof["selesai"] = g["datetime"].max().dt.date
    return prof.sort_values("jumlah", ascending=False).head(top).round(2)

print("Profil cluster terbesar (rentang waktu lebar = cluster spasial, bukan satu rangkaian susulan):")
display(cluster_profile(result))

## 10. Pengujian beberapa kombinasi parameter

Setiap kombinasi `(eps, MinPts)` pada grid dijalankan, lalu dibandingkan dengan metrik berikut:

| Metrik | Cara membaca |
|---|---|
| `n_clusters` | Terlalu banyak = terfragmentasi; 1-2 = kemungkinan menyatu |
| `noise_pct` | Persentase titik berlabel -1 |
| `largest_pct` | Porsi cluster terbesar terhadap seluruh data; nilai tinggi = indikasi chaining |
| `silhouette` | Dihitung hanya pada titik non-noise, jarak haversine, sampel acak. **Bukan ukuran mutlak untuk DBSCAN**: silhouette cenderung menyukai cluster padat berbentuk bulat dan mengabaikan noise, sehingga bentuk cluster memanjang di sepanjang zona subduksi bisa dinilai buruk walau bermakna secara geofisika |

Tidak ada metrik tunggal yang dapat menyatakan kombinasi "terbaik" di sini; yang dicari adalah kombinasi yang **masuk akal menurut beberapa metrik sekaligus** dan **dapat ditafsirkan** pada peta.

In [ ]:
def safe_silhouette(labels):
    m = labels != -1
    if len(np.unique(labels[m])) < 2:
        return np.nan
    try:
        return float(silhouette_score(X[m], labels[m], metric="haversine",
                                      sample_size=min(SILHOUETTE_SAMPLE, int(m.sum())), random_state=RANDOM_STATE))
    except Exception:
        return np.nan

grid_rows, grid_labels = [], {}
total = len(EPS_GRID_KM) * len(MINPTS_GRID)
i = 0
for m in MINPTS_GRID:
    for e in EPS_GRID_KM:
        i += 1
        db, lab, sec = fit_dbscan(e, m)
        grid_labels[(e, m)] = lab
        row = {"eps_km": e, "min_samples": m, **cluster_stats(lab)}
        row["core_pct"] = round(100 * len(db.core_sample_indices_) / N, 2)
        sil = safe_silhouette(lab)
        row["silhouette"] = round(sil, 3) if np.isfinite(sil) else np.nan
        row["detik"] = round(sec, 1)
        grid_rows.append(row)
        print(f"[{i:>2}/{total}] eps={e:>4} km, MinPts={m:>2} -> {row['n_clusters']:>3} cluster, noise {row['noise_pct']:>5}%, terbesar {row['largest_pct']:>5}%")

grid = pd.DataFrame(grid_rows)
grid.to_csv(f"{OUT_DIR}/perbandingan_parameter.csv", index=False)
print("\nTabel lengkap (diurutkan menurut MinPts lalu eps):")
display(grid[["eps_km", "min_samples", "n_clusters", "n_noise", "noise_pct", "largest_pct",
              "median_cluster_size", "core_pct", "silhouette", "detik"]])

In [ ]:
# --- Heatmap perbandingan ---
def heat(ax, col, title, fmt="{:.0f}", cmap="viridis"):
    pv = grid.pivot(index="min_samples", columns="eps_km", values=col).sort_index()
    im = ax.imshow(pv.values, cmap=cmap, aspect="auto")
    ax.set_xticks(range(pv.shape[1])); ax.set_xticklabels(pv.columns)
    ax.set_yticks(range(pv.shape[0])); ax.set_yticklabels(pv.index)
    ax.set_xlabel("eps (km)"); ax.set_ylabel("MinPts"); ax.set_title(title)
    lo, hi = np.nanmin(pv.values), np.nanmax(pv.values)
    for r in range(pv.shape[0]):
        for c in range(pv.shape[1]):
            v = pv.values[r, c]
            if np.isfinite(v):
                ax.text(c, r, fmt.format(v), ha="center", va="center", fontsize=8,
                        color="white" if (v - lo) / (hi - lo + 1e-9) < .6 else "black")
    plt.colorbar(im, ax=ax, shrink=.8)

fig, ax = plt.subplots(2, 2, figsize=(14, 8))
heat(ax[0, 0], "n_clusters", "Jumlah cluster")
heat(ax[0, 1], "noise_pct", "Noise (%)", "{:.0f}", "magma")
heat(ax[1, 0], "largest_pct", "Cluster terbesar (% dari seluruh data)", "{:.0f}", "magma")
heat(ax[1, 1], "silhouette", "Silhouette (non-noise, sampel)", "{:.2f}", "viridis")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/heatmap_perbandingan.png", dpi=130); plt.show()

In [ ]:
# --- Peta berdampingan beberapa kombinasi (ubah sesuai kebutuhan; hanya yang ada di grid yang digambar) ---
COMPARE_COMBOS = [(10, 10), (20, 10), (30, 10), (50, 10), (20, 4), (20, 20)]
combos = [c for c in COMPARE_COMBOS if c in grid_labels]

ncol = 3; nrow = int(np.ceil(len(combos) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(6 * ncol, 4.2 * nrow), squeeze=False)
for a in axes.ravel():
    a.axis("off")
for a, (e, m) in zip(axes.ravel(), combos):
    a.axis("on")
    plot_clusters(grid_labels[(e, m)], f"eps = {e} km, MinPts = {m}", a, s_cluster=3)
    a.set_xlabel(""); a.set_ylabel("")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/peta_perbandingan.png", dpi=130); plt.show()

### Menyaring kombinasi (bukan memilih "yang terbaik")

Sel berikut hanya **menyisihkan** kombinasi yang jelas bermasalah menurut kriteria yang Anda tetapkan sendiri di bawah (contoh nilai ada di sel; sesuaikan dengan tujuan analisis). Kombinasi yang lolos masih harus diperiksa pada peta dan ditafsirkan secara geofisika. Urutan berdasarkan silhouette hanya alat bantu pengurutan dengan keterbatasan yang dijelaskan di atas.

In [ ]:
# Kriteria penyaringan (contoh; ubah sesuai tujuan):
MIN_CLUSTERS   = 2      # minimal 2 cluster (satu cluster tidak informatif)
MAX_LARGEST_PCT = 50    # cluster terbesar tidak boleh memuat >50% seluruh data (hindari chaining)
MAX_NOISE_PCT   = 50    # noise tidak lebih dari 50%

ok = grid[(grid.n_clusters >= MIN_CLUSTERS) & (grid.largest_pct <= MAX_LARGEST_PCT) & (grid.noise_pct <= MAX_NOISE_PCT)]
print(f"Kriteria: cluster >= {MIN_CLUSTERS}, cluster terbesar <= {MAX_LARGEST_PCT}%, noise <= {MAX_NOISE_PCT}%")
if ok.empty:
    print("Tidak ada kombinasi yang lolos. Longgarkan kriteria atau perluas EPS_GRID_KM / MINPTS_GRID.")
else:
    print(f"{len(ok)} dari {len(grid)} kombinasi lolos (urut silhouette menurun; ini penyaring, bukan penentu terbaik):")
    display(ok.sort_values("silhouette", ascending=False)[["eps_km", "min_samples", "n_clusters", "noise_pct",
                                                             "largest_pct", "median_cluster_size", "silhouette"]])

### Visualisasi ulang untuk kombinasi yang Anda pilih

Isi `EPS_FINAL_KM` dan `MINPTS_FINAL` dengan kombinasi yang Anda anggap paling masuk akal setelah melihat tabel, heatmap, dan peta di atas, lalu jalankan sel. Default-nya sama dengan baseline.

In [ ]:
EPS_FINAL_KM = EPS_BASELINE      # <- ganti, mis. 20
MINPTS_FINAL = MINPTS_MAIN       # <- ganti, mis. 10

db_f, labels_f, sec = fit_dbscan(EPS_FINAL_KM, MINPTS_FINAL)
res_f = attach_labels(data, labels_f, db_f)
sf = cluster_stats(labels_f)
print(f"eps = {EPS_FINAL_KM} km, MinPts = {MINPTS_FINAL} -> {sf['n_clusters']} cluster, "
      f"{sf['n_noise']:,} noise ({sf['noise_pct']}%), cluster terbesar {sf['largest_pct']}%")

fig, ax = plt.subplots(figsize=(11, 6.5))
plot_clusters(labels_f, f"DBSCAN (eps = {EPS_FINAL_KM} km, MinPts = {MINPTS_FINAL})", ax)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/peta_cluster_final.png", dpi=130); plt.show()

display(cluster_profile(res_f))
res_f.to_csv(f"{OUT_DIR}/hasil_dbscan_final.csv", index=False)
print("Tersimpan:", sorted(os.listdir(OUT_DIR)))

if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/hasil_dbscan", "zip", OUT_DIR)
    files.download("/content/hasil_dbscan.zip")

## Catatan interpretasi

- **Nilai awal bukan nilai optimal.** ε baseline berasal dari titik siku k-distance dan MinPts dari aturan praktis serta ukuran data; keduanya baru valid sebagai titik awal. Kelayakan akhir ditentukan oleh perbandingan di Bagian 10 dan pemeriksaan peta.
- **Kepadatan global.** Satu pasang parameter untuk seluruh Indonesia akan lebih memihak zona yang padat. Bila hasil terlihat timpang (zona padat menyatu, zona jarang menjadi noise), itu keterbatasan DBSCAN, bukan kesalahan kode. HDBSCAN atau OPTICS (kepadatan variabel) adalah alternatif yang layak dibahas.
- **Ketergantungan pada preprocessing.** Hasil bergantung pada ambang kelengkapan `MC_THRESHOLD = 4.5`, periode 2010 ke atas, dan kotak wilayah. Mengubahnya di notebook preprocessing akan mengubah semua angka di sini.
- **Cluster adalah kelompok kepadatan spasial.** Rentang waktu cluster bisa belasan tahun; tafsir sebagai rangkaian susulan memerlukan analisis temporal tambahan.
- **Border point** dapat berpindah cluster bila urutan data berubah; core point dan jumlah cluster tidak terpengaruh. Data diurutkan menurut waktu agar hasil dapat direproduksi.